Purpose: Make a ratio file of preread and assay readsUses a table.txt with reader file names in columns 1 and 2 and calculates the ratio of the two plate reads.
table and reader files are tab delimited txt and csv format

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt




In [2]:
# ---------------- SET YOUR DATA FOLDER HERE ----------------
# Use the FULL absolute path to the folder containing table.txt and the CSVs.
# On Windows it'll look like: r"C:\Users\yourname\Documents\2026\08282020_GPR52_glosensor_S1P1_CYM_Lib\Data"
# On Mac/Linux it'll look like: "/Users/yourname/Documents/2026/08282020_GPR52_glosensor_S1P1_CYM_Lib/Data"
DATA_DIR = r"C:\Users\sbrown\Documents\2026\08282020_GPR52_glosensor_S1P1_CYM_Lib\Data"
# -------------------------------------------------------------

N_ROWS, N_COLS = 16, 24

# Sanity check before doing anything else
if not os.path.isdir(DATA_DIR):
    print(f"⚠ This folder does not exist: {DATA_DIR}")
    print(f"Current working directory is: {os.getcwd()}")
else:
    print(f"✓ Found folder: {DATA_DIR}")
    print("Contents:")
    for f in sorted(os.listdir(DATA_DIR)):
        print("  ", f)

✓ Found folder: C:\Users\sbrown\Documents\2026\08282020_GPR52_glosensor_S1P1_CYM_Lib\Data
Contents:
   .ipynb_checkpoints
   14626_20200831_A17116_001.csv
   14626_20200831_A17117_002.csv
   14626_20200831_A17118_003.csv
   14626_20200831_A17119_004.csv
   14626_20200831_A17120_005.csv
   14626_20200831_A17121_006.csv
   14627_20200831_A17116_001.csv
   14627_20200831_A17116_001_ratio.csv
   14627_20200831_A17117_002.csv
   14627_20200831_A17118_003.csv
   14627_20200831_A17119_004.csv
   14627_20200831_A17120_005.csv
   14627_20200831_A17121_006.csv
   table.txt


In [3]:
table = pd.read_csv(os.path.join(DATA_DIR, "table.txt"), sep="\t")
table

,Preread,Assayread
0,14626_20200831_A17116_001.csv,14627_20200831_A17116_001.csv
1,14626_20200831_A17117_002.csv,14627_20200831_A17117_002.csv
2,14626_20200831_A17118_003.csv,14627_20200831_A17118_003.csv
3,14626_20200831_A17119_004.csv,14627_20200831_A17119_004.csv
4,14626_20200831_A17120_005.csv,14627_20200831_A17120_005.csv
5,14626_20200831_A17121_006.csv,14627_20200831_A17121_006.csv


In [4]:
def well_id(row_idx, col_idx):
    return f"{chr(ord('A') + row_idx)}{col_idx + 1}"

records = []
missing = []
saved_files = []

for _, row in table.iterrows():
    file1 = row.iloc[0]   # Preread filename
    file2 = row.iloc[1]   # Assayread filename
    path1 = os.path.join(DATA_DIR, file1)
    path2 = os.path.join(DATA_DIR, file2)

    if not os.path.exists(path1):
        missing.append(file1)
        continue
    if not os.path.exists(path2):
        missing.append(file2)
        continue

    df1 = pd.read_csv(path1, header=None, nrows=N_ROWS)   # preread, first 16 rows
    df2 = pd.read_csv(path2, header=None, nrows=N_ROWS)   # assayread, first 16 rows

    # Calculate the ratio of the two DataFrames
    ratio = df2 / df1
    ratio_df = ratio.round(4)

    # ---- Save this pair's ratio matrix under a unique filename ----
    base_name = os.path.splitext(file2)[0]          # e.g. "14627_20200831_A17117_002"
    out_name = f"{base_name}_ratio.csv"
    out_path = os.path.join(DATA_DIR, out_name)
    ratio_df.to_csv(out_path, header=False, index=False)
    saved_files.append(out_name)

    # ---- Also keep a tidy long-format record across all pairs, for stats/plots ----
    for r in range(N_ROWS):
        for c in range(N_COLS):
            records.append({
                "pair": file2,
                "preread_file": file1,
                "well": well_id(r, c),
                "row_idx": r,
                "col_idx": c,
                "preread": df1.values[r, c],
                "assayread": df2.values[r, c],
                "ratio": ratio_df.values[r, c],
            })

results = pd.DataFrame(records)

if missing:
    print(f"⚠ Skipped file(s) not found: {missing}")

print(f"Loaded {results['pair'].nunique()} plate pair(s), {len(results)} wells total.")
print(f"\nSaved {len(saved_files)} ratio file(s) to {DATA_DIR}:")
for f in saved_files:
    print("  ", f)

Loaded 6 plate pair(s), 2304 wells total.

Saved 6 ratio file(s) to C:\Users\sbrown\Documents\2026\08282020_GPR52_glosensor_S1P1_CYM_Lib\Data:
   14627_20200831_A17116_001_ratio.csv
   14627_20200831_A17117_002_ratio.csv
   14627_20200831_A17118_003_ratio.csv
   14627_20200831_A17119_004_ratio.csv
   14627_20200831_A17120_005_ratio.csv
   14627_20200831_A17121_006_ratio.csv
